# Exercise: Neural Networks

## 1. Import the required libraries in the cell below

In [20]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

## 2. Read the data

In [21]:
housing = pd.read_csv('./data/sacramento_real_estate_transactions.csv')


## 3. Conduct Exploratory Data Analysis 

In [22]:
# Check data info
housing.info()


<class 'pandas.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   street     985 non-null    str    
 1   city       985 non-null    str    
 2   zip        985 non-null    int64  
 3   state      985 non-null    str    
 4   beds       985 non-null    int64  
 5   baths      985 non-null    int64  
 6   sq__ft     985 non-null    int64  
 7   type       985 non-null    str    
 8   sale_date  985 non-null    str    
 9   price      985 non-null    int64  
 10  latitude   985 non-null    float64
 11  longitude  985 non-null    float64
dtypes: float64(2), int64(5), str(5)
memory usage: 156.9 KB


In [23]:
# Check for missing values
housing.isna().sum()

street       0
city         0
zip          0
state        0
beds         0
baths        0
sq__ft       0
type         0
sale_date    0
price        0
latitude     0
longitude    0
dtype: int64

In [24]:
# Check data types
housing.dtypes

street           str
city             str
zip            int64
state            str
beds           int64
baths          int64
sq__ft         int64
type             str
sale_date        str
price          int64
latitude     float64
longitude    float64
dtype: object

In [25]:
housing.shape

(985, 12)

In [26]:
# Look at the Summary Stats.
housing.describe()

,zip,beds,baths,sq__ft,price,latitude,longitude
count,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000
mean,95750.697462,2.911675,1.776650,1312.918782,233715.951269,38.445121,-121.193371
std,85.176072,1.307932,0.895371,856.123224,139088.818896,5.103637,5.100670
min,95603.000000,0.000000,0.000000,-984.000000,-210944.000000,-121.503471,-121.551704
25%,95660.000000,2.000000,1.000000,950.000000,145000.000000,38.482704,-121.446119
50%,95762.000000,3.000000,2.000000,1304.000000,213750.000000,38.625932,-121.375799
75%,95828.000000,4.000000,2.000000,1718.000000,300000.000000,38.695589,-121.294893
max,95864.000000,8.000000,5.000000,5822.000000,884790.000000,39.020808,38.668433


In [27]:
# Cleaning Steps (if needed)
housing[housing['price']<=0]
housing = housing[housing['price']>0]

In [28]:
housing.shape

(984, 12)

In [30]:
housing['zip']=housing['zip'].astype(str)

In [31]:
housing['sale_date'] = housing['sale_date'].str.replace(' EDT ', ' ', regex=False)
housing['sale_date'] = pd.to_datetime(housing['sale_date'], format='%a %b %d %H:%M:%S %Y')

In [33]:
housing['type'].value_counts()

type
Residential     917
Condo            53
Multi-Family     13
Unkown            1
Name: count, dtype: int64

In [34]:
housing = housing[housing['type'] != 'Unkown']
housing['type'].value_counts()

type
Residential     917
Condo            53
Multi-Family     13
Name: count, dtype: int64

In [35]:
housing =housing[(housing['beds'] > 0) | (housing['baths'] > 0)]
housing =housing[ (housing['baths'] > 0)]
((housing['beds'] <= 0) | (housing['baths'] <= 0)).sum()

np.int64(0)

In [36]:
housing = housing[housing['sq__ft']>0]

In [37]:
housing.shape

(813, 12)

## 4. Split the Data & Pre-Processing
**Best Practice for Neural Network** Split your data before modeling into three part:

1. __The Training Set (The Textbook):__ This is the data the neural network actually looks at to learn. During the training loop, the network uses this data to adjust its **weights** and **biases** via backpropagation.

2. __The Validation Set (The Practice Quiz):__ This data evaluates the model *during* training to fine-tune settings. The model does not learn its weights from this data. Instead, the it uses the validation score to choose **hyperparameters** (like learning rate, number of layers, or node count) and to decide when to stop training (**early stopping**).


3. __The Test Set (The Final Exam):__ This data provides an unbiased, final evaluation of the model's real-world performance. This dataset is locked away until training is completely finished. It is only used *once* at the very end to see how well the network performs on truly unseen data.

__Typical Split Ratios__

Depending on the size of your total dataset, data scientists generally use these proportions:

| Dataset Size | Train | Validation | Test |
| :--- | :--- | :--- | :--- |
| **Traditional (Small/Medium Data)** | 60% | 20% | 20% |
| **Modern Deep Learning (Millions of Rows)** | 98% | 1% | 1% |

In [40]:
housing.columns

Index(['street', 'city', 'zip', 'state', 'beds', 'baths', 'sq__ft', 'type',
       'sale_date', 'price', 'latitude', 'longitude'],
      dtype='str')

In [41]:
# Select the Features and Target
X=housing[['sq__ft','beds', 'baths']]
y=housing['price']

In [42]:
# Split to Train Test & Validation
from sklearn.model_selection import train_test_split

X_train_all , X_test , y_train_all , y_test , = train_test_split(X,y,test_size=0.2,random_state=42)

X_train , X_val,y_train , y_val = train_test_split(X_train_all,y_train_all,test_size=0.2,random_state=42)

In [45]:
# IMfrom sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


## 5. Linear Regression

In [53]:
# Initialize LinearRegression model

lr = LinearRegression()


In [54]:
# Fit and check the Training Score

lr.fit(X_train_all,y_train_all)
lr.score(X_train_all,y_train_all)

0.5031033360412551

In [ ]:
# Predict and check the Testing Score


In [ ]:
# Evaluate the model
# Coefficient 

# Intercept

# RMSE


## 6. Build a Single Neural Network
Creating a model in `keras` entails a few steps:
1. Create your network topology
2. Compile your model
3. Fit your model
   
**Important Note on Regression:**
Because we are predicting a continuous value (house price), our network is performing regression, not classification. 
* Your **hidden layers** should generally use the `relu` activation function.
* Your **output layer** must have exactly **1 neuron** and **no activation function** (or `activation='linear'`) so it can predict any continuous number.
* When compiling, use `loss='mse'` (Mean Squared Error) and `metrics=['mae']` (Mean Absolute Error) so we can see the error in actual dollar amounts.

In [ ]:
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


## 7. Stacking Multiple Neurons

#### Shallow NN

In [ ]:
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


#### Deep Network

In [ ]:
# use multiple hidden layers with different neurons

# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


## 8. Activation Functions

In [ ]:
# from keras.layers import Activation 
# Note: import Activation if you use it as a standalone layer
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


## 9. Epochs

In [ ]:
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


## 10. Batch size (Chunks)

In [ ]:
# Use the validation
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


Try a different batch size. 

__Note:__ the best practice to have the value in powers of 2

In [ ]:
# Use the validation
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


## 11. History

**The Three Main Reasons to Use History**
  1. To Spot Overfitting
  2. To Visualise the Learning Curve
  3. To Know When to Stop Training

By saving the output of `model.fit()` to a variable (usually called `history`), Keras keeps a record of the training loss and validation loss at the end of each epoch. 

Plotting these two lines together helps us diagnose our model:
* If training loss goes down but validation loss goes up, the model is **overfitting** (memorizing the training data).
* If both are going down, the model is still learning.
* If both have flattened out, the model has converged and stopped learning.

In [ ]:
# Use the validation
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


In [ ]:
# Create a History Data Frame


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss


## 12. Early Stopping (callback)

- Prevent overfitting
- will not waste the computing power
- save time
  
It is hard to guess exactly how many epochs a model needs. If you train for too few epochs, the model underfits. If you train for too many, it overfits. 

An **Early Stopping Callback** acts as an automatic brake. It monitors the validation loss and automatically stops training once the validation score stops improving, even if you haven't reached your maximum number of epochs. It can also automatically restore the model's best weights!

In [ ]:
# configure EarlyStopping parameters


In [ ]:
# Use the validation
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


In [ ]:
# Create a History Data Frame


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss


## 13.Multi-Feature Neural Network

In [ ]:
# configure EarlyStopping parameters


In [ ]:
# Use the validation
# Step 1: Define



# Step 2: Compile 



# Step 3: Fit



In [ ]:
# Model Weights


In [ ]:
# Model Summary


In [ ]:
# Create a History Data Frame


In [ ]:
# Visualize Epochs vs. Loss for the Training Loss and Validation Loss
